In [4]:
import requests
from bs4 import BeautifulSoup

# 1. 검색할 URL 설정 (키워드: 노트북)
url = "https://search.naver.com/search.naver?where=news&query=노트북"

# 2. 네이버 서버에 "저 로봇 아니에요"라고 알려주는 'User-Agent' 헤더 설정
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# 3. 페이지 요청 및 HTML 가져오기
response = requests.get(url, headers=headers)

# 4. HTML 분석하기
soup = BeautifulSoup(response.text, "html.parser")

# 5. 뉴스 기사 제목이 들어있는 태그 찾기 
# (네이버 검색 결과에서 뉴스 제목은 보통 'news_tit'이라는 클래스 이름을 가집니다)
articles = soup.select(".news_tit")

# 6. 결과 출력하기
print("=== 네이버 뉴스 크롤링 결과 ===")
for idx, article in enumerate(articles, 1):
    title = article.text          # 기사 제목
    link = article["href"]        # 기사 링크
    
    print(f"{idx}. {title}")
    print(f"   링크: {link}\n")

=== 네이버 뉴스 크롤링 결과 ===


In [ ]:
# #함수 만들고 검색어 지정
# url 만드는 함수 하나 -> 그 url로 호출하는 함수하나
# -> 데이터 받아온거 저장? 하는 함수 하나 이 순서로 가는듯

In [6]:
import os
import sys
import time
import json
import urllib.request
import datetime

client_id = "YOUR_NAVER_CLIENT_ID"
client_secret = "YOUR_NAVER_CLIENT_SECRET"

def main():
    node = 'news'
    srcText = input('검색어를 입력하세요:')
    
    jsonResult = []

    jsonResponse = getNaverSearch(node, srcText, 1, 100)
    
    if jsonResponse is None:
        return
        
    total = jsonResponse['total']

    while ((jsonResponse != None) and (jsonResponse['display'] != 0)):
        for post in jsonResponse['items']:
            # 🌟 cnt를 넘겨주지 않고 데이터만 깔끔하게 넘깁니다.
            getPostData(post, jsonResult)
            
        start = jsonResponse['start'] + jsonResponse['display']
        if start > total or start > 1000:
            break
        jsonResponse = getNaverSearch(node, srcText, start, 100)
        
    print('전체 검색: %d 건' % total)

    with open('%s_naver_%s.json' % (srcText, node), 'w', encoding='utf8') as outfile:
        jsonFile = json.dumps(jsonResult, indent=4, sort_keys=True, ensure_ascii=False)
        outfile.write(jsonFile)

    # 🌟 [수정] 수동으로 세던 cnt 대신, 진짜 내 파일에 저장된 리스트의 개수(len)를 보여줍니다.
    print("오늘 자 진짜 저장된 데이터: %d 건" % len(jsonResult))
    print('%s_naver_%s.json SAVED' % (srcText, node))

def getRequestUrl(url):
    req = urllib.request.Request(url)
    req.add_header("X-Naver-Client-Id", client_id)
    req.add_header("X-Naver-Client-Secret", client_secret)
    try:
        response = urllib.request.urlopen(req)
        if response.getcode() == 200:
            print("[%s] Url Request Success" % datetime.datetime.now())
            return response.read().decode('utf-8')
    except Exception as e:
        print(e)
        return None
    
# 🌟 cnt 매개변수를 완전히 뺐습니다!
def getPostData(post, jsonResult):
    title = post['title']
    description = post['description']
    org_link = post['originallink']
    link = post['link']

    pDate = datetime.datetime.strptime(post['pubDate'], '%a, %d %b %Y %H:%M:%S +0900')
    
    if pDate.date() == datetime.datetime.now().date():
        pDate = pDate.strftime('%Y-%m-%d %H:%M:%S')
        
        # 🌟 저장할 때 순번이 굳이 필요하다면, 현재 내 주머니 크기에 +1을 해서 넣어주면 완벽합니다!
        current_no = len(jsonResult) + 1
        
        jsonResult.append({
            'cnt': current_no,  # 진짜 저장된 순서대로 예쁘게 1, 2, 3 번호가 매겨집니다.
            'title': title, 
            'description': description, 
            'org_link': org_link, 
            'link': link, 
            'pDate': pDate
        })
    return

def getNaverSearch(node, srcText, page_start, display):
    base = "https://openapi.naver.com/v1/search"
    node = "/%s.json" % node
    parameters = "?query=%s&start=%s&display=%s&sort=date" % (urllib.parse.quote(srcText), page_start, display)
    url = base + node + parameters
    responseDecode = getRequestUrl(url)
    if (responseDecode == None):
        return None
    else:
        return json.loads(responseDecode)

if __name__ == "__main__":
    main()

[2026-05-21 16:15:57.859021] Url Request Success
[2026-05-21 16:15:57.988437] Url Request Success
[2026-05-21 16:15:58.141861] Url Request Success
[2026-05-21 16:15:58.286799] Url Request Success
[2026-05-21 16:15:58.451790] Url Request Success
[2026-05-21 16:15:58.589758] Url Request Success
[2026-05-21 16:15:58.900059] Url Request Success
[2026-05-21 16:15:59.220431] Url Request Success
[2026-05-21 16:15:59.543498] Url Request Success
[2026-05-21 16:15:59.771948] Url Request Success
전체 검색: 1725677 건
오늘 자 진짜 저장된 데이터: 301 건
하이_naver_news.json SAVED


In [ ]:
#초급
import os
import sys
import time
import json
import urllib.request
import datetime

client_id = "YOUR_NAVER_CLIENT_ID"
client_secret = "YOUR_NAVER_CLIENT_SECRET"

#웹크롤링 메인함수
def main():
    node = 'news'
    srcText = input('검색어를 입력하세요:')
    cnt = 0
    jsonResult = []

#100번을 가져옴 그냥 정보자체를 100개 가져옴
    jsonResponse = getNaverSearch(node,srcText,1,100)
    total = jsonResponse['total']

#100개인지를 확인하기 위한 조건인듯 items 딕셔너리를 받아옴
    while((jsonResponse != None) and (jsonResponse['display']!=0)):
        for post in jsonResponse['items']:
            cnt += 1
            getPostData(post, jsonResult, cnt)
        start = jsonResponse['start'] + jsonResponse['display']
        if start > total or start > 1000:
            break
        jsonResponse = getNaverSearch(node, srcText, start, 100)
    print('전체 검색: %d 건' %total)

    with open('%s_naver_%s.json' % (srcText, node),'w',encoding = 'utf8') as outfile:
        jsonFile = json.dumps(jsonResult, indent =4, sort_keys = True, ensure_ascii = False)

        outfile.write(jsonFile)

    print("가져온 데이터: %d 건" %(cnt))
    print('%s_naver_%s.json SAVED' % (srcText,node))

def getRequestUrl(url):
    req = urllib.request.Request(url)
    req.add_header("X-Naver-Client-Id",client_id)
    req.add_header("X-Naver-Client-Secret",client_secret)
        
    #에러 났을때 로그를 남겨주는 부분 
    try:
        response = urllib.request.urlopen(req)
        if response.getcode() == 200:
            print("[%s] Url Request Success" % datetime.datetime.now())
            return response.read().decode('utf-8')
    except Exception as e:
        print(e)
        print("[%s] Error for URL:%s" % (datetime.datetime.now(),url))
        return None
    
def getPostData(post, jsonResult, cnt):
    title = post['title']
    description = post['description']
    org_link = post['originallink']
    link = post['link']

    pDate = datetime.datetime.strptime(post['pubDate'], '%a, %d %b %Y %H:%M:%S +0900')
    if pDate.date() == datetime.datetime.now().date():
        pDate = pDate.strftime('%Y-%m-%d %H:%M:%S')
        jsonResult.append({'cnt':cnt, 'title':title, 'description':description, 'org_link':org_link, 'link':org_link, 'pDate':pDate})
    return

#이게 그 네이버 뉴스 크롤링 api를 활용하는 방식이네 url 만드는법
#꼭 찾아서 들어가서 보고 플랫폼 마다 다름
def getNaverSearch(node,srcText, page_start, display):
    base = "https://openapi.naver.com/v1/search"
    node = "/%s.json" % node
    parameters = "?query=%s&start=%s&display=%s&sort=date" % (urllib.parse.quote(srcText),page_start,display)

    url = base + node + parameters
    responseDecode = getRequestUrl(url)

    if(responseDecode == None):
        return None
    else:
        return json.loads(responseDecode)
    

if __name__ == "__main__":
    main()

[2026-05-21 16:09:08.983699] Url Request Success
[2026-05-21 16:09:09.122936] Url Request Success
[2026-05-21 16:09:09.279350] Url Request Success
[2026-05-21 16:09:09.426461] Url Request Success
[2026-05-21 16:09:09.585488] Url Request Success
[2026-05-21 16:09:09.750727] Url Request Success
[2026-05-21 16:09:09.931943] Url Request Success
[2026-05-21 16:09:10.126365] Url Request Success
[2026-05-21 16:09:10.311118] Url Request Success
[2026-05-21 16:09:10.514718] Url Request Success
전체 검색: 12475016 건
가져온 데이터: 1000 건
삼성_naver_news.json SAVED


In [ ]:
#중급
import urllib.request
import json
import time
import datetime

client_id = "YOUR_NAVER_CLIENT_ID"
client_secret = "YOUR_NAVER_CLIENT_SECRET"

def main():
    node = 'news'
    a = {}
    for i in range(3):
        srcText = input('검색어를 입력하세요:')
        cnt = 0
        jsonResult = []

    #100번을 가져옴 그냥 정보자체를 100개 가져옴
        jsonResponse = getNaverSearch(node,srcText,1,100)
        total = jsonResponse['total']

        print('전체 검색: %d 건' %total)
        a[srcText] = total

    sorted_a = sorted(a.items(), key = lambda x: x[1] ,reverse = True )
    num = 0
    for k, t in sorted_a:
        print(f"{k}가 {t}개로 {num+1}번째")
        num += 1

def getRequestUrl(url):
    req = urllib.request.Request(url)
    req.add_header("X-Naver-Client-Id",client_id)
    req.add_header("X-Naver-Client-Secret",client_secret)
        
    #에러 났을때 로그를 남겨주는 부분 
    try:
        response = urllib.request.urlopen(req)
        if response.getcode() == 200:
            print("[%s] Url Request Success" % datetime.datetime.now())
            return response.read().decode('utf-8')
    except Exception as e:
        print(e)
        print("[%s] Error for URL:%s" % (datetime.datetime.now(),url))
        return None
    
def getNaverSearch(node,srcText, page_start, display):
    base = "https://openapi.naver.com/v1/search"
    node = "/%s.json" % node
    parameters = "?query=%s&start=%s&display=%s&sort=date" % (urllib.parse.quote(srcText),page_start,display)

    url = base + node + parameters
    responseDecode = getRequestUrl(url)

    if(responseDecode == None):
        return None
    else:
        return json.loads(responseDecode)
    
if __name__ == "__main__":
    main()

[2026-05-21 15:39:36.928795] Url Request Success
전체 검색: 1899829 건
[2026-05-21 15:39:38.152983] Url Request Success
전체 검색: 12474752 건
[2026-05-21 15:39:39.219493] Url Request Success
전체 검색: 3678559 건
삼성가 12474752개로 1번째
한화가 3678559개로 2번째
기아가 1899829개로 3번째


In [22]:
#중급
import urllib.request
import json
import time
import datetime

client_id = "YOUR_NAVER_CLIENT_ID"
client_secret = "YOUR_NAVER_CLIENT_SECRET"

def main():
    node = 'news'
    a = {}
    for i in range(3):
        srcText = input('검색어를 입력하세요:')
        cnt = 0
        jsonResult = []

    #100번을 가져옴 그냥 정보자체를 100개 가져옴
        jsonResponse = getNaverSearch(node,srcText,1,100)
        total = jsonResponse['total']

        print('전체 검색: %d 건' %total)
        a[srcText] = total

    sorted_a = sorted(a.items(), key = lambda x: x[1] ,reverse = True )
    num = 0
    for k, t in sorted_a:
        print(f"{k}이(가) {t}개로 {num+1}번째")
        num += 1

def getRequestUrl(url):
    req = urllib.request.Request(url)
    req.add_header("X-Naver-Client-Id",client_id)
    req.add_header("X-Naver-Client-Secret",client_secret)
        
    #에러 났을때 로그를 남겨주는 부분 
    try:
        response = urllib.request.urlopen(req)
        if response.getcode() == 200:
            print("[%s] Url Request Success" % datetime.datetime.now())
            return response.read().decode('utf-8')
    except Exception as e:
        print(e)
        print("[%s] Error for URL:%s" % (datetime.datetime.now(),url))
        return None
    
def getNaverSearch(node,srcText, page_start, display):
    base = "https://openapi.naver.com/v1/search"
    node = "/%s.json" % node
    parameters = "?query=%s&start=%s&display=%s&sort=date" % (urllib.parse.quote(srcText),page_start,display)

    url = base + node + parameters
    responseDecode = getRequestUrl(url)

    if(responseDecode == None):
        return None
    else:
        return json.loads(responseDecode)
    
if __name__ == "__main__":
    main()

[2026-05-21 17:24:02.839857] Url Request Success
전체 검색: 10596 건
[2026-05-21 17:24:06.502049] Url Request Success
전체 검색: 51416 건
[2026-05-21 17:24:09.784086] Url Request Success
전체 검색: 32811 건
인절미이(가) 51416개로 1번째
찹쌀떡이(가) 32811개로 2번째
두쫀쿠이(가) 10596개로 3번째


In [ ]:
#1주일간 정보량
import os
import sys
import time
import json
import urllib.request
import datetime

client_id = "YOUR_NAVER_CLIENT_ID"
client_secret = "YOUR_NAVER_CLIENT_SECRET"

#웹크롤링 메인함수
def main():
    node = 'news'
    srcText = input('검색어를 입력하세요:')
    cnt = 0
    jsonResult = []
    day7 = datetime.datetime.now().date()- datetime.timedelta(days=7)
    con7 = {}
    for i in range(7):
        con7[i]=0

#100번을 가져옴 그냥 정보자체를 100개 가져옴
    jsonResponse = getNaverSearch(node,srcText,1,100)
    total = jsonResponse['total']

#100개인지를 확인하기 위한 조건인듯 items 딕셔너리를 받아옴
    while((jsonResponse != None) and (jsonResponse['display']!=0)):
        for post in jsonResponse['items']:
            cnt += 1
            getPostData(post, jsonResult, cnt, con7, day7)
        start = jsonResponse['start'] + jsonResponse['display']
        if start > total or start > 1000:
            break
        jsonResponse = getNaverSearch(node, srcText, start, 100)
    print('전체 검색: %d 건' %total)

    with open('%s_naver_%s.json' % (srcText, node),'w',encoding = 'utf8') as outfile:
        jsonFile = json.dumps(jsonResult, indent =4, sort_keys = True, ensure_ascii = False)

        outfile.write(jsonFile)

    print("가져온 데이터: %d 건" %(cnt))
    print('%s_naver_%s.json SAVED' % (srcText,node))
    for i,k in con7.items():
        if i == 0:
            print(f"오늘 데이터수는 {k}")
        else:
            print(f"{i}일전 {k}")

def getRequestUrl(url):
    req = urllib.request.Request(url)
    req.add_header("X-Naver-Client-Id",client_id)
    req.add_header("X-Naver-Client-Secret",client_secret)
        
    #에러 났을때 로그를 남겨주는 부분 
    try:
        response = urllib.request.urlopen(req)
        if response.getcode() == 200:
            print("[%s] Url Request Success" % datetime.datetime.now())
            return response.read().decode('utf-8')
    except Exception as e:
        print(e)
        print("[%s] Error for URL:%s" % (datetime.datetime.now(),url))
        return None
    
def getPostData(post, jsonResult, cnt, con7, day7):
    title = post['title']
    description = post['description']
    org_link = post['originallink']
    link = post['link']

    pDate = datetime.datetime.strptime(post['pubDate'], '%a, %d %b %Y %H:%M:%S +0900')
    if pDate.date() > day7:
        cai = (datetime.datetime.now().date()-pDate.date()).days
        pDate = pDate.strftime('%Y-%m-%d %H:%M:%S')
        con7[cai] += 1
        jsonResult.append({'cnt':cnt, 'title':title, 'description':description, 'org_link':org_link, 'link':org_link, 'pDate':pDate})
    return

#이게 그 네이버 뉴스 크롤링 api를 활용하는 방식이네 url 만드는법
#꼭 찾아서 들어가서 보고 플랫폼 마다 다름
def getNaverSearch(node,srcText, page_start, display):
    base = "https://openapi.naver.com/v1/search"
    node = "/%s.json" % node
    parameters = "?query=%s&start=%s&display=%s&sort=date" % (urllib.parse.quote(srcText),page_start,display)

    url = base + node + parameters
    responseDecode = getRequestUrl(url)

    if(responseDecode == None):
        return None
    else:
        return json.loads(responseDecode)
    

if __name__ == "__main__":
    main()

[2026-05-21 17:21:16.288856] Url Request Success
[2026-05-21 17:21:16.397060] Url Request Success
[2026-05-21 17:21:16.748186] Url Request Success
[2026-05-21 17:21:16.889137] Url Request Success
[2026-05-21 17:21:16.997838] Url Request Success
[2026-05-21 17:21:17.087415] Url Request Success
[2026-05-21 17:21:17.200441] Url Request Success
[2026-05-21 17:21:17.295649] Url Request Success
[2026-05-21 17:21:17.399145] Url Request Success
[2026-05-21 17:21:17.741793] Url Request Success
전체 검색: 10596 건
가져온 데이터: 1000 건
두쫀쿠_naver_news.json SAVED
오늘 데이터수는 8
1일전 47
2일전 26
3일전 11
4일전 7
5일전 12
6일전 11


In [1]:
import os
import sys
import time
import json
import urllib.request
import datetime

client_id = "YOUR_NAVER_CLIENT_ID"
client_secret = "YOUR_NAVER_CLIENT_SECRET"

#웹크롤링 메인함수
def main():
    node = 'news'
    srcText = input('검색어를 입력하세요:')
    cnt = 0
    jsonResult = []
    t_24 = {}
    for i in range(24):
        t_24[i]=0
    

#100번을 가져옴 그냥 정보자체를 100개 가져옴
    jsonResponse = getNaverSearch(node,srcText,1,100)
    total = jsonResponse['total']

#100개인지를 확인하기 위한 조건인듯 items 딕셔너리를 받아옴
    while((jsonResponse != None) and (jsonResponse['display']!=0)):
        for post in jsonResponse['items']:
            cnt += 1
            getPostData(post, jsonResult, cnt, t_24)
        start = jsonResponse['start'] + jsonResponse['display']
        if start > total or start > 1000:
            break
        jsonResponse = getNaverSearch(node, srcText, start, 100)
    print('전체 검색: %d 건' %total)

    with open('%s_naver_%s.json' % (srcText, node),'w',encoding = 'utf8') as outfile:
        jsonFile = json.dumps(jsonResult, indent =4, sort_keys = True, ensure_ascii = False)

        outfile.write(jsonFile)

    print("가져온 데이터: %d 건" %(cnt))
    print('%s_naver_%s.json SAVED' % (srcText,node))
    num = 0
    for i,k in t_24.items():
        if num % 5 != 0:
            print(f'{i}시 {k}개',end=' ')
        else:
            print(f'{i}시는 {k}개\n')
        num += 1


def getRequestUrl(url):
    req = urllib.request.Request(url)
    req.add_header("X-Naver-Client-Id",client_id)
    req.add_header("X-Naver-Client-Secret",client_secret)
        
    #에러 났을때 로그를 남겨주는 부분 
    try:
        response = urllib.request.urlopen(req)
        if response.getcode() == 200:
            print("[%s] Url Request Success" % datetime.datetime.now())
            return response.read().decode('utf-8')
    except Exception as e:
        print(e)
        print("[%s] Error for URL:%s" % (datetime.datetime.now(),url))
        return None
    
def getPostData(post, jsonResult, cnt, t_24):
    title = post['title']
    description = post['description']
    org_link = post['originallink']
    link = post['link']

    pDate = datetime.datetime.strptime(post['pubDate'], '%a, %d %b %Y %H:%M:%S +0900')
    pDate = pDate.strftime('%Y-%m-%d %H:%M:%S')
    t_pDate = pDate.split(' ')
    t_hour = t_pDate[1].split(':')[0]
    t_hour = int(t_hour)
    t_24[t_hour] +=1


    jsonResult.append({'cnt':cnt, 'title':title, 'description':description, 'org_link':org_link, 'link':org_link, 'pDate':pDate})
    return

#이게 그 네이버 뉴스 크롤링 api를 활용하는 방식이네 url 만드는법
#꼭 찾아서 들어가서 보고 플랫폼 마다 다름
def getNaverSearch(node,srcText, page_start, display):
    base = "https://openapi.naver.com/v1/search"
    node = "/%s.json" % node
    parameters = "?query=%s&start=%s&display=%s&sort=date" % (urllib.parse.quote(srcText),page_start,display)

    url = base + node + parameters
    responseDecode = getRequestUrl(url)

    if(responseDecode == None):
        return None
    else:
        return json.loads(responseDecode)
    

if __name__ == "__main__":
    main()

[2026-05-22 10:31:06.077918] Url Request Success
[2026-05-22 10:31:06.224513] Url Request Success
[2026-05-22 10:31:06.388934] Url Request Success
[2026-05-22 10:31:06.536500] Url Request Success
[2026-05-22 10:31:06.694714] Url Request Success
[2026-05-22 10:31:06.873537] Url Request Success
[2026-05-22 10:31:07.099916] Url Request Success
[2026-05-22 10:31:07.408324] Url Request Success
[2026-05-22 10:31:07.610526] Url Request Success
[2026-05-22 10:31:07.815667] Url Request Success
전체 검색: 1900274 건
가져온 데이터: 1000 건
기아_naver_news.json SAVED
0시는 10개

1시 2개 2시 2개 3시 1개 4시 3개 5시는 8개

6시 24개 7시 35개 8시 80개 9시 111개 10시는 57개

11시 58개 12시 24개 13시 49개 14시 67개 15시는 97개

16시 123개 17시 103개 18시 54개 19시 16개 20시는 20개

21시 24개 22시 20개 23시 12개 

In [ ]:
#고급
import os
import sys
import time
import json
import urllib.request
import datetime

client_id = "YOUR_NAVER_CLIENT_ID"
client_secret = "YOUR_NAVER_CLIENT_SECRET"

#웹크롤링 메인함수
def main():
    node = 'news'
    srcText = input('검색어를 입력하세요:')
    cnt = 0
    jsonResult = []
    day7 = datetime.datetime.now().date()- datetime.timedelta(days=7)
    con7 = {}
    for i in range(8):
        con7[i]=0

    t_24 = {}
    for i in range(24):
        t_24[i]=0

#그냥 정보자체를 100개 가져옴
    jsonResponse = getNaverSearch(node,srcText,1,100)
    total = jsonResponse['total']

#100개인지를 확인하기 위한 조건인듯 items 딕셔너리를 받아옴
    while((jsonResponse != None) and (jsonResponse['display']!=0)):
        for post in jsonResponse['items']:
            cnt += 1
            getPostData(post, jsonResult, cnt, con7, day7, t_24)
        start = jsonResponse['start'] + jsonResponse['display']
        if start > total or start > 1000:
            break
        jsonResponse = getNaverSearch(node, srcText, start, 100)
    print('전체 검색: %d 건' %total)

    with open('%s_naver_%s.json' % (srcText, node),'w',encoding = 'utf8') as outfile:
        jsonFile = json.dumps(jsonResult, indent =4, sort_keys = True, ensure_ascii = False)

        outfile.write(jsonFile)

    print("가져온 데이터: %d 건" %(cnt))
    print('%s_naver_%s.json SAVED' % (srcText,node))
    for i,k in con7.items():
        if i == 0:
            print(f"오늘 데이터수는 {k}")
        elif i == 7:
            print(f"{i}일전 {k}\n")
        else:
            print(f"{i}일전 {k}",end = " ,")
    num = 1
    for i,k in t_24.items():
        if num % 4 != 0:
            print(f'{i}시 {k}개',end=' ')
        else:
            print(f'{i}시는 {k}개')
            print('\n')
        num += 1

def getRequestUrl(url):
    req = urllib.request.Request(url)
    req.add_header("X-Naver-Client-Id",client_id)
    req.add_header("X-Naver-Client-Secret",client_secret)
        
    #에러 났을때 로그를 남겨주는 부분 
    try:
        response = urllib.request.urlopen(req)
        if response.getcode() == 200:
            print("[%s] Url Request Success" % datetime.datetime.now())
            return response.read().decode('utf-8')
    except Exception as e:
        print(e)
        print("[%s] Error for URL:%s" % (datetime.datetime.now(),url))
        return None
    
def getPostData(post, jsonResult, cnt, con7, day7, t_24):
    title = post['title']
    description = post['description']
    org_link = post['originallink']
    link = post['link']

    pDate = datetime.datetime.strptime(post['pubDate'], '%a, %d %b %Y %H:%M:%S +0900')
    pDate_1 = pDate.strftime('%Y-%m-%d %H:%M:%S')
    t_pDate = pDate_1.split(' ')
    t_hour = t_pDate[1].split(':')[0]
    t_hour = int(t_hour)
    t_24[t_hour] +=1
    
    if pDate.date() > day7:
        cai = (datetime.datetime.now().date()-pDate.date()).days
        pDate = pDate.strftime('%Y-%m-%d %H:%M:%S')
        con7[cai] += 1
        jsonResult.append({'cnt':cnt, 'title':title, 'description':description, 'org_link':org_link, 'link':org_link, 'pDate':pDate})
    return

#이게 그 네이버 뉴스 크롤링 api를 활용하는 방식이네 url 만드는법
#꼭 찾아서 들어가서 보고 플랫폼 마다 다름
def getNaverSearch(node,srcText, page_start, display):
    base = "https://openapi.naver.com/v1/search"
    node = "/%s.json" % node
    parameters = "?query=%s&start=%s&display=%s&sort=date" % (urllib.parse.quote(srcText),page_start,display)

    url = base + node + parameters
    responseDecode = getRequestUrl(url)

    if(responseDecode == None):
        return None
    else:
        return json.loads(responseDecode)
    

if __name__ == "__main__":
    main()


[2026-05-22 09:04:28.834762] Url Request Success
[2026-05-22 09:04:28.957956] Url Request Success
[2026-05-22 09:04:29.098625] Url Request Success
[2026-05-22 09:04:29.229675] Url Request Success
[2026-05-22 09:04:29.365341] Url Request Success
[2026-05-22 09:04:29.610640] Url Request Success
[2026-05-22 09:04:29.815446] Url Request Success
[2026-05-22 09:04:30.020310] Url Request Success
[2026-05-22 09:04:30.224929] Url Request Success
[2026-05-22 09:04:30.446135] Url Request Success
전체 검색: 505905 건
가져온 데이터: 1000 건
키움 히어로즈_naver_news.json SAVED
오늘 데이터수는 30
1일전 525 ,2일전 445 ,3일전 0 ,4일전 0 ,5일전 0 ,6일전 0 ,7일전 0

0시 28개 1시 5개 2시 0개 3시는 5개


4시 1개 5시 9개 6시 6개 7시는 10개


8시 5개 9시 8개 10시 6개 11시는 10개


12시 5개 13시 17개 14시 11개 15시는 14개


16시 13개 17시 20개 18시 63개 19시는 244개


20시 223개 21시 224개 22시 53개 23시는 20개


